# Chapter `1.4` - Multi-modal Agent

## Setup

### Module imports

In [16]:
# Basic utils.
import time
import base64

from os import getenv
from dotenv import load_dotenv

# o/p formatting
from pprint import pprint
from IPython.display import display, Markdown

# i/o modules
import io
from ipywidgets import FileUpload

from tqdm import tqdm
import sounddevice as sd
from scipy.io.wavfile import write

# LC modules
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

from langchain.messages import AIMessage
from langchain.messages import HumanMessage

from langgraph.checkpoint.memory import InMemorySaver

### **Gemini** API setup

In [5]:
load_dotenv()

GOOGLE_API_KEY = getenv("GOOGLE_API_KEY")
GEMINI_API_MODEL = getenv("GEMINI_API_MODEL")

model = ChatGoogleGenerativeAI(model=GEMINI_API_MODEL, api_key=GOOGLE_API_KEY)
agent = create_agent(
    model=model,
    system_prompt="You are a multi-modal agent that can handle different modalities of input data from the user."
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


## Text input

In [6]:
question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

Markdown(response['messages'][-1].content)

The Moon does not have a capital city. It is a natural satellite of Earth and is not a country or political entity with its own government or administrative divisions.

## Image input

### Uploading an image file
![Trie Node Code Snippet](../../pics/trieNode.png)

> This is a `.png` format image.

In [9]:
uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [11]:
pprint(uploader.value)

({'content': <memory at 0x000002650EEBB400>,
  'last_modified': datetime.datetime(2025, 4, 19, 16, 1, 59, 916000, tzinfo=datetime.timezone.utc),
  'name': 'trieNode.png',
  'size': 65213,
  'type': 'image/png'},)


### Encoding the image to `base64`
> This is done to aid comprehension for the **LLM**.

In [12]:
# Fetch the FIRST uploaded file
uploaded_file = uploader.value[0]

# This is a memoryview object
content_mv = uploaded_file["content"]

# Convert memoryview object -> bytes object
img_bytes = content_mv.tobytes()

# Now, perform base64 encoding
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

### Prompting

In [13]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "What do you make of this image?"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

Markdown(response['messages'][-1].content)

The image shows a code snippet written in what appears to be Java. It defines a class named `Node` with two instance variables, `one` and `zero`, both of type `Node`. The class also has a constructor `public Node()` which initializes both `one` and `zero` to `null`.

This structure is commonly used in data structures like binary trees or linked lists, where each node can have references to other nodes. The names `one` and `zero` might suggest a binary tree where each node has at most two children, representing branches for a "0" or "1" path, or potentially a boolean-like structure.

## Audio input

### Recording configuration

In [20]:
duration = 5  # in seconds
sample_rate = 44100

print("🎤 Now, recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)

# Progress bar for the duration
for _ in tqdm(range(duration * 15)):  # update 10x per second
    time.sleep(0.1)
sd.wait()

print("☑️ Recording finished.")

🎤 Now, recording...


100%|██████████| 75/75 [00:07<00:00,  9.82it/s]

☑️ Recording finished.


### Encoding the audio file

In [21]:
# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

# Encode the audio file
aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

### Prompting

In [22]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "What you can make of this audio file? Gimme a report."},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

Markdown(response['messages'][-1].content)

This audio file contains a short spoken phrase. The content appears to be a question in English: **"What is dead may never die?"**

The audio quality is clear, and the speech is easily understandable. There are no background noises or other discernible sounds.